In [13]:
import json
import re
import random

In [14]:
last_case = None

In [15]:
import glob

data = []
for filename in glob.glob("./data/to_be_cleaned_alt*.json"):
    with open(filename) as f:
        data.extend(json.load(f))

In [16]:
print(len(data))

194000


In [18]:
# from tqdm import tqdm
# import json
# import re

# all = []
# bad_all = []
# # with open("all_fine_tuning12_ext.json") as f:
# #     data = json.load(f)
# for item in tqdm(data, desc="Processing cases"):
#     text = item['text']
#     cleaned_text = re.sub(r"^```json\s*|\s*```$", "", text.strip())
#     last_case = [item['cnr'], cleaned_text]
#     if item['cnr'] in bad_cases:
#         print(f"Skipping bad case: {item['cnr']}")
#         continue
#     # try:
#     info = {
#         "CNR": item['cnr'],
#         # "type_of_application": re.search(r'"type_of_application"\s*:\s*"([^"]+)"', cleaned_text).group(1),
#         # "application_for_withdrawal": re.search(r'"application_for_withdrawal"\s*:\s*"([^"]+)"', cleaned_text).group(1),
#         # "accused_details": get_accused_details(cleaned_text),
#         # "past_criminal_records": "Yes" if re.search(r'"past_criminal_records"\s*:\s*"(?P<value>Yes|No|None)"', cleaned_text).group('value') == 'Yes' else 'No',
#         # "statutes": get_statutes(cleaned_text),
#         # "precedents": get_precedents(cleaned_text),
#         # "details_of_the_incident": re.search(r'"details of the incident"\s*:\s*"([^"]*?)"', cleaned_text).group(1),
#         # # "arguments": {
#         # #     "supporting": re.search(r'"Arguments"\s*:\s*{\s*"Arguments supporting the application"\s*:\s*"(.+?)",\s*"Arguments opposing the application"\s*:\s*"(.+?)"\s*}', cleaned_text, re.DOTALL|re.IGNORECASE).group(1),
#         # #     "opposing": re.search(r'"Arguments"\s*:\s*{\s*"Arguments supporting the application"\s*:\s*"(.+?)",\s*"Arguments opposing the application"\s*:\s*"(.+?)"\s*}', cleaned_text, re.DOTALL|re.IGNORECASE).group(2),
#         # # },
#         # "arguments": extract_arguments(cleaned_text),
#         # "outcome": {
#         #     "status": re.search(r'"outcome"\s*:\s*\{[^}]*?"status"\s*:\s*"([^"]+)"', text).group(1),
#         #     "bail_conditions": [] if "not" in re.search(r'"outcome"\s*:\s*\{[^}]*?"status"\s*:\s*"([^"]+)"', text).group(1) else get_bail_conditions(cleaned_text),
#         # },
        
#         "reasoning": get_reasoning(cleaned_text),
#         "date_of_arrest": get_date_of_arrest(cleaned_text),
#         "date_of_judgment": get_date_of_judgement(cleaned_text),        
#     }
#     all.append(info)
#     # except Exception as e:
#     #     bad_all.append({
#     #         "cnr": item['cnr'],
#     #         "text": cleaned_text,
#     #         "error": str(e)
#     #     })
            
#     json.dump(bad_all, open("./data/gondogol_hoche.json", "w"), indent=4)

#     all.sort(key=lambda x: len(x['reasoning'])+sum(len(detail) for detail in x['details_of_the_incident'])+len(x['arguments']['supporting'])+len(x['arguments']['opposing']), reverse=True)

# with open("./data/chaat_k_saaf_kar_diye.json", "w") as f:
#     json.dump(all, f, indent=4)


In [19]:
from datetime import datetime

def normalize_date(date_str):
    if not date_str or not isinstance(date_str, str):
        return date_str

    date_str = date_str.strip()

    for fmt in ("%d-%m-%Y", "%d/%m/%Y", "%Y-%m-%d", "%d.%m.%Y",
                "%d.%m.%y", "%d/%m/%y", "%d-%m-%y"):
        try:
            parsed = datetime.strptime(date_str, fmt)
            return parsed.strftime("%d-%m-%Y")
        except ValueError:
            continue

    # If none match, return original
    return date_str


In [22]:
data_sorted = sorted(data, key=lambda x: sum(c.isalnum() for c in x['text']), reverse=True)

In [23]:
import re
import json
from tqdm import tqdm

def extract_field(field_name, text):
    """Extract the value of a given JSON key using regex."""
    pattern = rf'"{field_name}"\s*:\s*"((?:[^"\\]|\\.)*?)"'
    match = re.search(pattern, text, re.DOTALL)
    return match.group(1).encode().decode('unicode_escape') if match else ""

# Optional: Remove wrapping ```json ... ```
def clean_text(text):
    return re.sub(r"^```json\s*|\s*```$", "", text.strip())


all_cleaned = []
bad_all = []

for item in tqdm(data, desc="Processing cases"):
    try:
        raw_text = item['text']
        cleaned = clean_text(raw_text)

        parsed = {
            "CNR": item["cnr"],
            "case": extract_field("case", cleaned),
            "outcome": extract_field("outcome", cleaned),
            "reasoning": extract_field("reasoning", cleaned),
            "date_of_arrest": normalize_date(extract_field("date_of_arrest", cleaned)),
            "date_of_judgement": normalize_date(extract_field("date_of_judgement", cleaned)),
        }

        all_cleaned.append(parsed)
    except Exception as e:
        bad_all.append({
            "cnr": item['cnr'],
            "text": item['text'],
            "error": str(e)
        })

# Save the results
with open("./data/chaat_k_saaf_kar_diye.json", "w") as f:
    json.dump(all_cleaned, f, indent=4)

with open("./data/gondogol_hoche.json", "w") as f:
    json.dump(bad_all, f, indent=4)


Processing cases: 100%|██████████| 194000/194000 [01:25<00:00, 2269.95it/s]


In [29]:
def convert_case_dict(src):
    # --- 1. CASE SUMMARY ---
    lines = []
    # Bail/Application type
    app_type = src.get("type_of_application", "Unknown")
    lines.append(f"Applicant applied for {app_type}.")

    # Withdrawal
    lines.append(f"Is it a withdrawal application? {src.get('application_for_withdrawal','Unknown')}.")

    # Accused details
    if "accused_details" in src:
        ages = []
        healths = []
        for accused in src["accused_details"]:
            if "Age" in accused and accused["Age"] is not None:
                ages.append(str(accused["Age"]))
            hi = accused.get("health_info", {})
            health_desc = hi.get("description", None)
            if health_desc and health_desc.lower() != "none" and health_desc.lower() != "n/a":
                healths.append(str(health_desc))
        if ages:
            lines.append(f"Age of the accused is {', '.join(ages)} years.")
        else:
            lines.append("Age of the accused is unknown.")
        if healths:
            lines.append(f"Health issues for the accused are {', '.join(healths)}.")
        else:
            lines.append("Health issues for the accused are None.")

    # Past criminal record
    past_criminal_records = src.get("past_criminal_records", "Unknown")
    if past_criminal_records.lower() in ("n/a", "none"):
        past_criminal_records = "No"
    lines.append(f"There are {'no ' if past_criminal_records.lower() == 'no' else ''}past criminal records of the accused.")

    # Statutes
    statutes = src.get("statutes", [])
    lines.append("Statutes mentioned in the judgement are [" + ", ".join(statutes) + "].")

    # Precedents
    precedents = src.get("precedents", [])
    if precedents and len(precedents) > 0:
        lines.append("Precedents mentioned in the judgement are " + ", ".join(precedents) + ".")
    else:
        lines.append("Precedents mentioned in the judgement are None.")

    # Incident details
    incident = src.get("details_of_the_incident", "Not specified.")
    lines.append(f"Details of the incident are {incident}")

    # --- 2. Arguments ---
    arguments = src.get("arguments", {})
    lines.append("Arguments supporting the bail application are " + arguments.get("supporting", "None provided.") )
    lines.append("Arguments opposing the bail application are " + arguments.get("opposing", "None provided.") )

    case = '\n'.join(lines)

    # --- 3. OUTCOME ---
    outcome_obj = src.get("outcome", {})
    status = outcome_obj.get('status', 'Unknown')
    bail_conditions = outcome_obj.get('bail_conditions', [])
    if bail_conditions:
        bail_conds = " ".join(bail_conditions)
    else:
        bail_conds = "None."
    outcome = f"The outcome of the case is {status}. The bail conditions are {bail_conds}"

    # --- 4. REASONING ---
    reasoning = src.get("reasoning", None)
    if reasoning is None:
        reasoning_out = "No reasoning specified."
    elif isinstance(reasoning, list):
        reasoning_out = " ".join(reasoning)
    else:
        reasoning_out = reasoning

    # --- 5. Dates ---
    date_of_arrest = src.get("date_of_arrest", None) or "Unknown"
    date_of_judgement = src.get("date_of_judgment", None) or "Unknown"

    # --- Final dictionary ---
    tgt = {
        "CNR": src.get("CNR", "Unknown"),
        "case": case,
        "outcome": outcome,
        "reasoning": reasoning_out,
        "date_of_arrest": date_of_arrest,
        "date_of_judgement": date_of_judgement
    }
    return tgt


# # ------------ Example usage: -------------
# example_dict = {
#     "CNR": "HCBM010178302018",
#     "type_of_application": "Regular-Bail",
#     "application_for_withdrawal": "No",
#     "accused_details": [
#         {
#             "Age": 20,
#             "health_info": {
#                 "description": "None",
#                 "score": 0
#             }
#         },
#         {
#             "Age": 27,
#             "health_info": {
#                 "description": "None",
#                 "score": 0
#             }
#         }
#     ],
#     "past_criminal_records": "No",
#     "statutes": [
#         "Section 302 IPC", "Section 307 IPC", "Section 324 IPC", "Section 143 IPC"
#     ],
#     "precedents": [],
#     "details_of_the_incident": "The incident occurred ...",
#     "arguments": {
#         "supporting": "No role of assault attributed to the applicants...",
#         "opposing": "The sequence of events has to be considered..."
#     },
#     "outcome": {
#         "status": "Bail granted",
#         "bail_conditions": [
#             "Applicants to be released on bail...",
#             "Applicants to attend police station once a month..."
#         ]
#     },
#     "reasoning": [
#         "Applicants are not assigned any role...", "Prosecution invoked Section 149 of IPC..."
#     ],
#     "date_of_arrest": None,
#     "date_of_judgment": None
# }

# print(convert_case_dict(example_dict))


In [30]:
new_data = []
with open("./data/final_cleaned.json", "r") as f:
    old_data = json.load(f)
    for item in old_data:
        new_item = convert_case_dict(item)
        new_data.append(new_item)

with open("./data/final_cleaned_alt_1.json", "w") as f:
    json.dump(new_data, f, indent=4)

In [31]:
with open("./data/final_cleaned_alt_1.json", "r") as f1, open("./data/final_cleaned_alt_2.json", "r") as f2:
    data1 = json.load(f1)
    data2 = json.load(f2)

merged_data = data1 + data2

with open("./data/final_cleaned_alt.json", "w") as fout:
    json.dump(merged_data, fout, indent=4)

In [32]:
merged_data[0]

{'CNR': 'KLHC010003852010',
 'case': "Applicant applied for Anticipatory-Bail.\nIs it a withdrawal application? No.\nAge of the accused is 40 years.\nHealth issues for the accused are None.\nThere are no past criminal records of the accused.\nStatutes mentioned in the judgement are [Section 341 IPC, Section 376 IPC, Section 506 IPC, Section 511 IPC, Section 366 r/w 34 IPC].\nPrecedents mentioned in the judgement are None.\nDetails of the incident are The accused are charged with offences related to a maid servant who was allegedly physically assaulted and raped by the accused. The maid servant was working in the house of the first accused and was allegedly tied up and assaulted by the accused while the first accused's wife was not present.\nArguments supporting the bail application are The allegations made against the first accused are baseless and false. The de facto complainant had a motive to falsely implicate the first accused in the crime. The first accused lost money from the hou

In [40]:
sample = """
12th June, 2020
02/19.01.2021
03/17.01.2020
6/28.08.2019
22nd November, 2019
26-03-2019
7th November, 2019
07/09.03.2016
02/15.09.2020
05/05.07.2019
20-09-2017
14-07-2021
2/06.10.2016
08-07-2014
3/26.10.2016
12th October, 2020
01st June, 2020
04/04.04.2017
02/04.10.2018
02/11.02.2014
14-12-2012
04/10.1.2020
16-08-2016
16-07-2018
29-11-2019
02/31.07.2020
03-03-2015
27-01-2020
2/09.07.2020
05/09.05.2017
10-08-2021
25-07-2018
31-07-2018
25-02-2014
05-09-2018
2/14.9.2016
10-09-2020
02/10.07.2020
26-11-2018
29-10-2018
6/09.01.2014
28-02-2017
22-08-2019
16-07-2019
02/30.10.2015
02/02.07.2020
21-02-2018
5/09.02.2017
02/28.05.2018
02/13.05.2019
15th October, 2020
02/3-9-2020
03/08.07.2019
2/ 13.05.2013
06-06-2018
28-03-2019
11-08-2020
05/29.08.2019
24-06-2020
27-07-2018
03/12.01.2021
29-01-2015
05/20.03.2013
5/15.02.2018
2/20.08.2020
10/14.02.2018
04/12.08.2014
21-10-2016
03-11-2020
10th August, 2016
22-07-2019
02/01.07.2020
23rd day of March, 2020
"""

import re
from rapidfuzz import process, fuzz

def clean_date(original_date):
    month_no = {
        "jan": "01", "feb": "02", "mar": "03", "apr": "04",
        "may": "05", "jun": "06", "jul": "07",
        "aug": "08", "sep": "09", "oct": "10",
        "nov": "11", "dec": "12",
        "january": "01", "february": "02", "march": "03",
        "april": "04", "may": "05", "june": "06",
        "july": "07", "august": "08", "september": "09",
        "october": "10", "november": "11", "december": "12"
    }
    months = list(month_no.keys())
    date_str = ""
    original_date = original_date.strip()
    if re.match(r"^\d{2}-\d{2}-\d{4}$", original_date):
        return original_date
    parts = re.findall(r"\d+", original_date)
    if len(parts) == 4:
        date_str = f"{parts[1]}-{parts[2]}-{parts[3]}"
    elif len(parts) == 2:
        month_match = process.extractOne(original_date.lower(), months, scorer=fuzz.partial_ratio)
        if month_match and month_match[1] > 80:
            date_str = f"{parts[0]}-{month_no[month_match[0]]}-{parts[1]}"
    return date_str

# Example usage:
for line in sample.strip().splitlines():
    print(f"Original: {line.strip()} | Parsed Date: {clean_date(line)}")


Original: 12th June, 2020 | Parsed Date: 12-06-2020
Original: 02/19.01.2021 | Parsed Date: 19-01-2021
Original: 03/17.01.2020 | Parsed Date: 17-01-2020
Original: 6/28.08.2019 | Parsed Date: 28-08-2019
Original: 22nd November, 2019 | Parsed Date: 22-11-2019
Original: 26-03-2019 | Parsed Date: 26-03-2019
Original: 7th November, 2019 | Parsed Date: 7-11-2019
Original: 07/09.03.2016 | Parsed Date: 09-03-2016
Original: 02/15.09.2020 | Parsed Date: 15-09-2020
Original: 05/05.07.2019 | Parsed Date: 05-07-2019
Original: 20-09-2017 | Parsed Date: 20-09-2017
Original: 14-07-2021 | Parsed Date: 14-07-2021
Original: 2/06.10.2016 | Parsed Date: 06-10-2016
Original: 08-07-2014 | Parsed Date: 08-07-2014
Original: 3/26.10.2016 | Parsed Date: 26-10-2016
Original: 12th October, 2020 | Parsed Date: 12-10-2020
Original: 01st June, 2020 | Parsed Date: 01-06-2020
Original: 04/04.04.2017 | Parsed Date: 04-04-2017
Original: 02/04.10.2018 | Parsed Date: 04-10-2018
Original: 02/11.02.2014 | Parsed Date: 11-02-2

In [44]:
def clean_date(original_date):
    month_no = {
        "jan": "01", "feb": "02", "mar": "03", "apr": "04",
        "may": "05", "jun": "06", "jul": "07",
        "aug": "08", "sep": "09", "oct": "10",
        "nov": "11", "dec": "12",
        "january": "01", "february": "02", "march": "03",
        "april": "04", "may": "05", "june": "06",
        "july": "07", "august": "08", "september": "09",
        "october": "10", "november": "11", "december": "12"
    }
    months = list(month_no.keys())
    date_str = ""
    original_date = original_date.strip()
    if re.match(r"^\d{2}-\d{2}-\d{4}$", original_date):
        return original_date
    parts = re.findall(r"\d+", original_date)
    if len(parts) == 4:
        date_str = f"{parts[1]}-{parts[2]}-{parts[3]}"
    elif len(parts) == 2:
        month_match = process.extractOne(original_date.lower(), months, scorer=fuzz.partial_ratio)
        if month_match and month_match[1] > 80:
            date_str = f"{parts[0]}-{month_no[month_match[0]]}-{parts[1]}"
    else:
        date_str = original_date
    return date_str

aug_data = []
with open("./data/final_cleaned_augmented.json", 'r') as f:
    aug_data = json.load(f)
        

In [48]:
print(json.dumps(aug_data[1], indent=2, ensure_ascii=False))

{
  "case": "Applicant applied for Regular-Bail.\nIs it a withdrawal application? No.\nAge of the accused is 30 years.\nHealth issues for the accused are None.\nThere are no past criminal records of the accused.\nStatutes mentioned in the judgement are [Section 143 IPC, Section 147 IPC, Section 148 IPC, Section 459 IPC, Section 354 IPC, Section 323 IPC, Section 324 IPC, Section 120 B IPC, Section 212 IPC, Section 364-A IPC, Section 302 IPC, Section 149 IPC].\nPrecedents mentioned in the judgement are None.\nDetails of the incident are The accused are charged with offences related to a hawala money transaction that resulted in the abduction and murder of Dawood. The prosecution alleges that the accused were involved in a hawala racket and that Dawood was abducted and murdered by the accused.\nArguments supporting the bail application are About two months are over since the arrest of the petitioners. There is no reason why the petitioners should be detained further. Advocate Sri.Sunny Ma

In [49]:
for data in tqdm(aug_data, desc="Cleaning dates"):
    data['date_of_arrest'] = clean_date(data['date_of_arrest'])
    data['date_of_judgement'] = clean_date(data['date_of_judgement'])

# with open("./data/final_cleaned_augmented_date.json", "w", encoding="utf-8") as f:
#     json.dump(aug_data, f, ensure_ascii=False, indent=4)

Cleaning dates: 100%|██████████| 208292/208292 [00:02<00:00, 94813.10it/s] 
